# English–Nepali Joint Training with XLM-R + CNN

This notebook implements a research-grade bilingual NLP model for Nepali (Devanagari) and English using XLM-RoBERTa, CNN trigram features, contrastive joint training, and UMAP/t-SNE visualization.

In [ ]:
!pip install torch transformers umap-learn scikit-learn matplotlib

## Imports and Device Setup

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import XLMRobertaTokenizer, XLMRobertaModel
import matplotlib.pyplot as plt
import umap
from sklearn.manifold import TSNE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Tokenizer

In [ ]:

tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_text(texts, max_len=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )


## Model Definition

In [ ]:

class NepaliCNNTransformer(nn.Module):
    def __init__(self, num_classes, filters=128, kernel_size=3, freeze_encoder=False):
        super().__init__()

        self.encoder = XLMRobertaModel.from_pretrained("xlm-roberta-base")

        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

        hidden_size = self.encoder.config.hidden_size
        padding = kernel_size // 2

        self.conv1d = nn.Conv1d(
            in_channels=hidden_size,
            out_channels=filters,
            kernel_size=kernel_size,
            padding=padding
        )

        fusion_dim = hidden_size + filters
        self.fc1 = nn.Linear(fusion_dim, 64)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, input_ids, attention_mask, return_embedding=False):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state

        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (token_embeddings * mask).sum(dim=1)
        mean_pool = mean_pool / torch.clamp(mask.sum(dim=1), min=1e-9)

        if return_embedding:
            return mean_pool

        x = token_embeddings.permute(0, 2, 1)
        conv_out = F.relu(self.conv1d(x))
        global_max_pool = torch.max(conv_out, dim=2)[0]

        fused = torch.cat([mean_pool, global_max_pool], dim=1)
        x = F.relu(self.fc1(fused))
        x = self.dropout(x)
        logits = self.classifier(x)
        return logits


## Contrastive Loss for English–Nepali Joint Training

In [ ]:

def contrastive_loss(emb_en, emb_ne, temperature=0.05):
    emb_en = F.normalize(emb_en, dim=1)
    emb_ne = F.normalize(emb_ne, dim=1)

    sim = torch.matmul(emb_en, emb_ne.T) / temperature
    labels = torch.arange(sim.size(0)).to(sim.device)

    loss_en = F.cross_entropy(sim, labels)
    loss_ne = F.cross_entropy(sim.T, labels)
    return (loss_en + loss_ne) / 2


## English–Nepali Parallel Data

In [ ]:

english_texts = [
    "Nepal is a beautiful country",
    "The weather is very nice today"
]

nepali_texts = [
    "नेपाल एक सुन्दर देश हो",
    "आज मौसम धेरै राम्रो छ"
]


## Joint Training Loop

In [ ]:

inputs_en = tokenize_text(english_texts)
inputs_ne = tokenize_text(nepali_texts)

inputs_en = {k: v.to(device) for k, v in inputs_en.items()}
inputs_ne = {k: v.to(device) for k, v in inputs_ne.items()}

model = NepaliCNNTransformer(num_classes=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

model.train()
for epoch in range(5):
    optimizer.zero_grad()

    emb_en = model(**inputs_en, return_embedding=True)
    emb_ne = model(**inputs_ne, return_embedding=True)

    loss = contrastive_loss(emb_en, emb_ne)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1} | Joint Loss: {loss.item():.4f}")


## Extract Sentence Embeddings

In [ ]:

model.eval()

all_sentences = english_texts + nepali_texts
labels_lang = ["English"] * len(english_texts) + ["Nepali"] * len(nepali_texts)

inputs = tokenize_text(all_sentences)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    embeddings = model(**inputs, return_embedding=True)

embeddings = embeddings.cpu().numpy()


## UMAP Visualization

In [ ]:

reducer = umap.UMAP(n_neighbors=5, min_dist=0.3, random_state=42)
emb_2d = reducer.fit_transform(embeddings)

plt.figure(figsize=(7,6))
for lang in set(labels_lang):
    idx = [i for i,l in enumerate(labels_lang) if l == lang]
    plt.scatter(emb_2d[idx,0], emb_2d[idx,1], label=lang)

for i, txt in enumerate(all_sentences):
    plt.annotate(txt, (emb_2d[i,0], emb_2d[i,1]), fontsize=8)

plt.legend()
plt.title("UMAP: English–Nepali Sentence Alignment")
plt.show()


## t-SNE Visualization

In [ ]:

tsne = TSNE(n_components=2, perplexity=5, random_state=42)
emb_2d_tsne = tsne.fit_transform(embeddings)

plt.figure(figsize=(7,6))
plt.scatter(emb_2d_tsne[:,0], emb_2d_tsne[:,1])

for i, txt in enumerate(all_sentences):
    plt.annotate(txt, (emb_2d_tsne[i,0], emb_2d_tsne[i,1]), fontsize=8)

plt.title("t-SNE: English–Nepali Sentence Embeddings")
plt.show()
